In [1]:
import pandas as pd 

df = pd.read_csv("/workspace/kor_med_opendataset/snuh_ClinicalQA/train.csv")

row_sample = df.head(1).to_dict(orient="records")[0]
row_sample

import ast
def get_snuh_ClinicalQA_prompt(row):
    # Parse options safely (assuming it's a string representation of a dict)

    # options 파싱 및 포맷팅
    options_dict = ast.literal_eval(row['options'])
    formatted_options = "\n".join([
        f"{key.replace('option_', '').upper()}) {value}"
        for key, value in sorted(options_dict.items())
    ])

    # Qwen에 최적화된 한국어 프롬프트
    snuh_ClinicalQA_prompt = f"""당신은 숙련된 내과 전문의입니다. 다음 임상 사례를 바탕으로 환자의 주소증에 대한 **가장 가능성 높은 단일 원인**을 판단하세요.

    제시된 보기 중에서 정답을 선택하고, 반드시 **유효한 JSON 형식**으로만 응답하세요.  
    - "answer" 필드는 "**X) 보기 텍스트**" 형식을 정확히 따르세요 (예: "C) 원발성 담즙성 담관염").  
    - "explanation" 필드는 근거를 설명하세요.  
    - JSON 외의 어떤 추가 텍스트도 출력하지 마세요.

    질문: {row['question']}

    보기:
    {formatted_options}
    """

    return snuh_ClinicalQA_prompt

## 

## sean0042_KorMedMCQA

In [2]:
import pandas as pd
from glob import glob 
# 데이터 합치기 
for domain in ['doctor', 'nurse', 'dentist', 'pharm']:
    dev = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_dev.csv")
    fewshot = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_fewshot.csv")
    train = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_train.csv")
    test = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_test.csv")
    all = pd.concat([dev, fewshot, train, test])
    # 중복치 제거
    all = all.drop_duplicates(subset=['question'])
    # all.to_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_all.csv", index=False)

In [3]:
import sys 
sys.path.append("/workspace")
import pandas as pd 
from src.qa_prompt import get_sean0042_KorMedMCQA_prompt
# prompt 만들기
df = pd.read_csv("/workspace/kor_med_opendataset/sean0042_KorMedMCQA/doctor/doctor_all.csv")


row = df.head(1).to_dict(orient="records")[0]

prompt = get_sean0042_KorMedMCQA_prompt(row)

print(prompt)

당신은 의사입니다.
질문:
광역시 소재 대학병원에 소속된 내과 전문의 A가 콜레라 환자를 진단했다. A가 할 조치는?

보기:
1) 병원장에게 보고
2) 광역시장에게 신고
3) 질병관리청장에게 신고
4) 관할 보건소장에게 신고
5) 보건복지부장관에게 신고

질문을 분석하고, 제시된 보기 중에서 한개의 보기를 선택하고, 근거를 설명해.
- JSON 외의 어떤 추가 텍스트도 출력하지 마세요.- 반드시 다음 JSON 형식으로만 답하세요:
{"answer":"보기","explanation":"한국어 근거"}



## aihub_전문_의학지식_데이터

In [4]:
result_root_dir = "/workspace/kor_med_opendataset/aihub_전문_의학지식_데이터"

In [5]:
from IPython.display import display
import simdjson as json 
from glob import glob 
import pandas as pd 
js_files = glob(f"{result_root_dir}/**/02.라벨링데이터/*.json", recursive=False)

def json_load(file_path):
    with open(file_path, 'r') as f:
        return json.load(f)

js_dict = {}
for id, js_file in enumerate(js_files):
    js_dict[id] = json_load(js_file)

df = pd.DataFrame(js_dict)
df = df.T

In [6]:
import pandas as pd 

df = pd.read_csv("/workspace/kor_med_opendataset/aihub_전문_의학지식_데이터/라벨링데이터.csv")

서술형_df = df[df["q_type"] == 3]
객관식형_df = df[df["q_type"] == 1]
단답형_df = df[df["q_type"] == 2]

서술형_df.to_csv("/workspace/kor_med_opendataset/aihub_전문_의학지식_데이터/라벨링데이터_서술형.csv", index=False)
객관식형_df.to_csv("/workspace/kor_med_opendataset/aihub_전문_의학지식_데이터/라벨링데이터_객관식.csv", index=False)
단답형_df.to_csv("/workspace/kor_med_opendataset/aihub_전문_의학지식_데이터/라벨링데이터_단답형.csv", index=False)

print(len(서술형_df))
print(len(객관식형_df))
print(len(단답형_df))

1754
10382
1620


## aihub_필수의료_의학지식_데이터

In [1]:
result_root_dir = "/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터"

In [ ]:
from IPython.display import display
import simdjson as json 
from glob import glob 
import pandas as pd 
js_files = glob(f"{result_root_dir}/**/02.라벨링데이터/*.json", recursive=False)

def json_load(file_path):
    with open(file_path, 'r') as f:
        return json.load(f)

js_dict = {}
for id, js_file in enumerate(js_files):
    js_dict[id] = json_load(js_file)

df = pd.DataFrame(js_dict)
df = df.T
# df.to_csv("/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터/라벨링데이터.csv", index=False)
df['q_type'].value_counts()

In [7]:
import pandas as pd 

df = pd.read_csv("/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터/라벨링데이터.csv")

서술형_df = df[df["q_type"] == 3]
객관식형_df = df[df["q_type"] == 1]
단답형_df = df[df["q_type"] == 2]

서술형_df.to_csv("/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터/라벨링데이터_서술형.csv", index=False)
객관식형_df.to_csv("/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터/라벨링데이터_객관식.csv", index=False)
단답형_df.to_csv("/workspace/kor_med_opendataset/aihub_필수의료_의학지식_데이터/라벨링데이터_단답형.csv", index=False)

print(len(서술형_df))
print(len(객관식형_df))
print(len(단답형_df))

1618
14028
1634
